In [ ]:
from datasets import load_dataset
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
import random
import pandas as pd
import datetime

In [ ]:
def generate_dict(x, lang):
    lang_ds = load_dataset("wikimedia/wikipedia", "20231101.{lang}".format(lang=lang))
    print('dataset ' + '{lang}'.format(lang=lang) + ' loaded')
    # Consider returning number of rows in datasets
    corpus_lang = lang_ds['train']['text'][0:x]
    # print(corpus_lang)
    # Convert to big string
    string_lang = " ".join(corpus_lang)
    tokenized_lang= word_tokenize(string_lang)
    print("{lang}".format(lang=lang) + " after tokenisation")
    print("{lang}".format(lang=lang), " : ", str(len(tokenized_lang)), " words")
    # Remove punctuation, proper nouns and numerical entries
    cleaned_lang = [word for word in tokenized_lang if ((len(word) > 1) and (not(word[0].isupper())) and (not(word.isnumeric())))]
    print("{lang}".format(lang=lang) + " after data cleaning")
    print("{lang}".format(lang=lang), " : ", str(len(cleaned_lang)), " words")
    dict_lang = {}
    for word in cleaned_lang:
        if word not in dict_lang:
            dict_lang[word] = 0
        dict_lang[word] += 1
    return dict_lang


In [ ]:
af = generate_dict(100, 'af')
nl = generate_dict(100, 'nl')


In [ ]:
# print(type(nl))

In [ ]:
corpus_af_nl = {'afrikaans' : af, 'nederlands' : nl}


In [ ]:
# print(corpus_af_nl)

In [ ]:
# Creating test function using small sample data, before using wikipedia data

# corpus_1 = {'en' : {'cat' : 2, 'dog' : 5, 'lizard' : 7},
#           'fr' : {'chat' : 4, 'chien' : 2, 'lizard' : 10}}

# Function takes a word and a corpus and returns its language
def identify_language(given_word, corpus):
    current_language = 'unknown'
    highest_frequency = 0
    for language_name, word_frequencies in corpus.items():
        if given_word in word_frequencies:
            if word_frequencies[given_word] > highest_frequency:
                current_language = language_name
                highest_frequency = word_frequencies[given_word]

    return current_language, highest_frequency




In [ ]:
identify_language('was', corpus_af_nl)

In [ ]:
identify_language('grootste', corpus_af_nl)

In [ ]:
identify_language('geïntegreerd', corpus_af_nl)

In [ ]:
identify_language('hier', corpus_af_nl)

In [ ]:
identify_language('daarvoor', corpus_af_nl)

In [ ]:
identify_language('gaan', corpus_af_nl)

In [ ]:
identify_language('onder', corpus_af_nl)

In [ ]:
# Dutch sequence of 4 words from wikipedia
# 'levens van meer dan'
print(identify_language('levens', corpus_af_nl))
print(identify_language('van', corpus_af_nl))
print(identify_language('meer', corpus_af_nl))
print(identify_language('dan', corpus_af_nl))

In [ ]:
# Afrikaans sentence taken from wikipedia (sequence of four words for comparison, some shared words)
# 'ook daardie jaar die'

print(identify_language('ook', corpus_af_nl))
print(identify_language('daardie', corpus_af_nl))
print(identify_language('jaar', corpus_af_nl))
print(identify_language('die', corpus_af_nl))

In [ ]:
# [word for word in tokenized_al_speer_no_punc if not word[0].isupper()]
# [word for word in tokenized_afrika_no_proper_nouns if not word.isnumeric()]

In [139]:
def generate_test_phrase(x: int, lang: str, phrase_length: int):
    lang_ds = load_dataset("wikimedia/wikipedia", "20231101.{lang}".format(lang=lang))
    # print('dataset loaded')
    corpus_lang = lang_ds['train']['text'][0:x]
    # Convert to big string
    string_lang = " ".join(corpus_lang)
    # print(str(len(string_lang)) + ' words in string_lang')
    tokenized_lang= word_tokenize(string_lang)
    # print(str(len(tokenized_lang)) + ' words in token lang')
    cleaned_lang = [word for word in tokenized_lang if (len(word) > 1)]
    # print(str(len(cleaned_lang)) + 'words in clean lang')
    no_proper_nouns_lang = [word for word in cleaned_lang if not word[0].isupper()]
    no_numbers_lang = [word for word in no_proper_nouns_lang if not word.isnumeric()]
    random_number = random.randint(0,(len(no_numbers_lang) - phrase_length))
    return no_numbers_lang[random_number : random_number+phrase_length]

In [ ]:
afrikaans_test_phrase_1 = generate_test_phrase(100, 'af', 10)
print(afrikaans_test_phrase_1)

In [ ]:
[identify_language(word, corpus_af_nl) for word in afrikaans_test_phrase_1]

In [ ]:
nederlands_test_phrase_1 = generate_test_phrase(100, 'nl', 10)
print(nederlands_test_phrase_1)

In [ ]:
[identify_language(word, corpus_af_nl) for word in nederlands_test_phrase_1]

In [ ]:
result = [identify_language(word, corpus_af_nl) for word in nederlands_test_phrase_1]
# print(result)


# print(result[0][0])
language_occurrence = [lang for lang, freq in result]
s = pd.Series(language_occurrence)
print(s.mode())


In [ ]:
def sentence_language_identifier(sentence: list[str], corpus):
    list_languages = [identify_language(word, corpus) for word in sentence]
    language_occurrence = [lang for lang, freq in list_languages]
    lang_occ_series = pd.Series(language_occurrence)
    modal_lang = lang_occ_series.mode()
    if modal_lang.empty:
        return 'unknown'
    else:
        return str(modal_lang)[1:16]

In [ ]:
nl1 = generate_test_phrase(10, 'nl', 2)
sentence_language_identifier(nl1, corpus_af_nl)

In [ ]:
af1 = generate_test_phrase(1, 'af', 2)
sentence_language_identifier(af1, corpus_af_nl)

In [ ]:
searching_bug = generate_test_phrase(2 ,'nl', 2)
print(searching_bug)

In [145]:
# Testing dutch sentences

results = {}

for corpus_size in range(1,6):
    for i in range(1,11):
        test_sentence = generate_test_phrase(corpus_size ,'nl', i)
        start_time = datetime.datetime.now()
        predicted_language = sentence_language_identifier(test_sentence, corpus_af_nl)
        end_time = datetime.datetime.now()
        results['Corpus size'] = corpus_size
        results['Sentence_length'] = len(test_sentence)
        results['Language'] = predicted_language
        results['Time taken'] = end_time - start_time
      # print('Corpus made of ', article_number, ' articles')
        # print('Sentence made of ', sentence_number, ' words')
        # print('Language is', result.upper())
        # print('time taken: ', time_taken)
        print(results)



{'Corpus size': 1, 'Sentence_length': 1, 'Language': '    nederlands\n', 'Time taken': datetime.timedelta(microseconds=616)}
{'Corpus size': 1, 'Sentence_length': 2, 'Language': '     afrikaans\n', 'Time taken': datetime.timedelta(microseconds=552)}
{'Corpus size': 1, 'Sentence_length': 3, 'Language': '    nederlands\n', 'Time taken': datetime.timedelta(microseconds=558)}
{'Corpus size': 1, 'Sentence_length': 4, 'Language': '    nederlands\n', 'Time taken': datetime.timedelta(microseconds=575)}
{'Corpus size': 1, 'Sentence_length': 5, 'Language': '    nederlands\n', 'Time taken': datetime.timedelta(microseconds=593)}
{'Corpus size': 1, 'Sentence_length': 6, 'Language': '    nederlands\n', 'Time taken': datetime.timedelta(microseconds=582)}
{'Corpus size': 1, 'Sentence_length': 7, 'Language': '    nederlands\n', 'Time taken': datetime.timedelta(microseconds=561)}
{'Corpus size': 1, 'Sentence_length': 8, 'Language': '    nederlands\n', 'Time taken': datetime.timedelta(microseconds=587)}


In [146]:
# Testing Afrikaans sentences

results = {}

for corpus_size in range(1,6):
    for i in range(1,11):
        test_sentence = generate_test_phrase(corpus_size ,'af', i)
        start_time = datetime.datetime.now()
        predicted_language = sentence_language_identifier(test_sentence, corpus_af_nl)
        end_time = datetime.datetime.now()
        results['Corpus size'] = corpus_size
        results['Sentence_length'] = len(test_sentence)
        results['Language'] = predicted_language
        results['Time taken'] = end_time - start_time
      # print('Corpus made of ', article_number, ' articles')
        # print('Sentence made of ', sentence_number, ' words')
        # print('Language is', result.upper())
        # print('time taken: ', time_taken)
        print(results)



{'Corpus size': 1, 'Sentence_length': 1, 'Language': '    afrikaans\nd', 'Time taken': datetime.timedelta(microseconds=572)}
{'Corpus size': 1, 'Sentence_length': 2, 'Language': '    afrikaans\nd', 'Time taken': datetime.timedelta(microseconds=585)}
{'Corpus size': 1, 'Sentence_length': 3, 'Language': '    afrikaans\nd', 'Time taken': datetime.timedelta(microseconds=640)}
{'Corpus size': 1, 'Sentence_length': 4, 'Language': '    afrikaans\nd', 'Time taken': datetime.timedelta(microseconds=644)}
{'Corpus size': 1, 'Sentence_length': 5, 'Language': '    afrikaans\nd', 'Time taken': datetime.timedelta(microseconds=648)}
{'Corpus size': 1, 'Sentence_length': 6, 'Language': '    afrikaans\nd', 'Time taken': datetime.timedelta(microseconds=606)}
{'Corpus size': 1, 'Sentence_length': 7, 'Language': '    afrikaans\nd', 'Time taken': datetime.timedelta(microseconds=641)}
{'Corpus size': 1, 'Sentence_length': 8, 'Language': '    afrikaans\nd', 'Time taken': datetime.timedelta(microseconds=570)}


KeyError: 'data'

In [153]:
from nltk.classify.textcat import TextCat

In [154]:
# refactoring existing LID tool to include TextCat results in the output, testing both models on the same data

# Testing Afrikaans sentences
tc = TextCat()
# language = tc.guess_language("sample sentence")
# print(language)

results = {}

for corpus_size in range(1,6):
    for i in range(1,11):
        test_sentence = generate_test_phrase(corpus_size ,'af', i)
        start_time_rule_based = datetime.datetime.now()
        predicted_language_rule_based = sentence_language_identifier(test_sentence, corpus_af_nl)
        end_time_rule_based = datetime.datetime.now()
        results['Corpus size'] = corpus_size
        results['Sentence_length'] = len(test_sentence)
        results['Language (RB)'] = predicted_language_rule_based
        results['Time taken (RB)'] = end_time_rule_based - start_time_rule_based
        test_sentence_joined = " ".join(test_sentence)
        start_time_textcat = datetime.datetime.now()
        predicted_language_textcat = tc.guess_language(test_sentence_joined)
        end_time_textcat = datetime.datetime.now()
        results['Language (TC)'] = predicted_language_textcat
        results['Time taken (TC)'] = end_time_textcat - start_time_rule_based
        print(results)



{'Corpus size': 1, 'Sentence_length': 1, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelta(microseconds=670), 'Language (TC)': 'mam', 'Time taken (TC)': datetime.timedelta(microseconds=5274)}
{'Corpus size': 1, 'Sentence_length': 2, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelta(microseconds=618), 'Language (TC)': 'afr', 'Time taken (TC)': datetime.timedelta(microseconds=55044)}
{'Corpus size': 1, 'Sentence_length': 3, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelta(microseconds=840), 'Language (TC)': 'afr', 'Time taken (TC)': datetime.timedelta(microseconds=48599)}
{'Corpus size': 1, 'Sentence_length': 4, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelta(microseconds=684), 'Language (TC)': 'afr', 'Time taken (TC)': datetime.timedelta(microseconds=114090)}
{'Corpus size': 1, 'Sentence_length': 5, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelta(microseconds

In [220]:
# Testing Dutch sentences
tc = TextCat()

result_dict = {}
results_list = []

for corpus_size in range(1,6):
    for i in range(1,11):
        test_sentence = generate_test_phrase(corpus_size ,'nl', i)
        start_time_rule_based = datetime.datetime.now()
        predicted_language_rule_based = sentence_language_identifier(test_sentence, corpus_af_nl)
        end_time_rule_based = datetime.datetime.now()
        result_dict['Corpus size'] = corpus_size
        result_dict['Sentence_length'] = len(test_sentence)
        result_dict['Language (RB)'] = predicted_language_rule_based
        result_dict['Time taken (RB)'] = end_time_rule_based - start_time_rule_based
        test_sentence_joined = " ".join(test_sentence)
        # print(test_sentence_joined)
        start_time_textcat = datetime.datetime.now()
        predicted_language_textcat = tc.guess_language(test_sentence_joined)
        end_time_textcat = datetime.datetime.now()
        result_dict['Language (TC)'] = predicted_language_textcat
        result_dict['Time taken (TC)'] = end_time_textcat - start_time_textcat
        results_list.append(result_dict)

print(results_list)



[{'Corpus size': 5, 'Sentence_length': 10, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelta(microseconds=642), 'Language (TC)': 'fri', 'Time taken (TC)': datetime.timedelta(microseconds=283053)}, {'Corpus size': 5, 'Sentence_length': 10, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelta(microseconds=642), 'Language (TC)': 'fri', 'Time taken (TC)': datetime.timedelta(microseconds=283053)}, {'Corpus size': 5, 'Sentence_length': 10, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelta(microseconds=642), 'Language (TC)': 'fri', 'Time taken (TC)': datetime.timedelta(microseconds=283053)}, {'Corpus size': 5, 'Sentence_length': 10, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelta(microseconds=642), 'Language (TC)': 'fri', 'Time taken (TC)': datetime.timedelta(microseconds=283053)}, {'Corpus size': 5, 'Sentence_length': 10, 'Language (RB)': '    afrikaans\nd', 'Time taken (RB)': datetime.timedelt

In [221]:
df = pd.DataFrame(results_list)

In [219]:
print(df)

    Corpus size  Sentence_length     Language (RB)        Time taken (RB)  \
0             5               10      nederlands\n 0 days 00:00:00.000594   
1             5               10      nederlands\n 0 days 00:00:00.000594   
2             5               10      nederlands\n 0 days 00:00:00.000594   
3             5               10      nederlands\n 0 days 00:00:00.000594   
4             5               10      nederlands\n 0 days 00:00:00.000594   
5             5               10      nederlands\n 0 days 00:00:00.000594   
6             5               10      nederlands\n 0 days 00:00:00.000594   
7             5               10      nederlands\n 0 days 00:00:00.000594   
8             5               10      nederlands\n 0 days 00:00:00.000594   
9             5               10      nederlands\n 0 days 00:00:00.000594   
10            5               10      nederlands\n 0 days 00:00:00.000594   
11            5               10      nederlands\n 0 days 00:00:00.000594   

In [184]:
print(type(results))

<class 'dict'>


In [185]:
print(results)

{'Corpus size': 5, 'Sentence_length': 10, 'Language (RB)': '    nederlands\n', 'Time taken (RB)': datetime.timedelta(microseconds=593), 'Language (TC)': 'eng', 'Time taken (TC)': datetime.timedelta(microseconds=198849)}


0
